# Shen et al. (2008): neuer Datensatz
Bereinigter Abschnitt aus „Neu machen – JupyterLab.pdf“, Seiten 15–29. Die älteren Datensätze und wiederholten Eingabeabfragen sind entfernt.

**Quelle:** SDSS, VizieR J/AJ/135/928/table1.
[Datentabelle](https://vizier.cfa.harvard.edu/viz-bin/VizieR?-source=J/AJ/135/928/table1)

**Csigma**: aus stellaren Absorptionslinien bestimmte, aperturkorrigierte Geschwindigkeitsdispersion in km/s.
**logM**: logarithmische viriale Massenschätzung aus AGN-Spektren. Die individuelle Masse wird nicht durch Einsetzen der stellaren Dispersion in die M–σ-Relation erzeugt.

## Ergebnisse aus deinem PDF
Die Zahlen sind aus der PDF-Ausgabe übernommen, nicht hier neu aus Rohdaten berechnet.

960 Katalogeinträge, **903 gültige Wertepaare**, 57 ausgeschlossene Einträge.

| Größe | Mittelwert | Median | Stichprobenstandardabweichung |
|---|---:|---:|---:|
| σ [km/s] | 175,93 | 165,90 | 61,121 |
| M [Sonnenmassen] | 7,0289 × 10⁷ | 3,7154 × 10⁷ | 1,2666 × 10⁸ |
| log₁₀(M/M☉) [dex] | 7,5842 | 7,57 | 0,44855 |

Anpassung: log₁₀(M/M☉) = 7,702 + 1,477 log₁₀(σ/200).
Residualstreuung: **0,394 dex** mit Nenner N und **0,395 dex** mit Nenner N−2.

## Erneut auswerten
Zellen von oben nach unten ausführen. Internet wird für den Download benötigt. Ergebnisse und Grafiken werden im Unterordner Shen2008_Auswertung gespeichert.


In [ ]:
%pip -q install astroquery astropy numpy scipy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from astroquery.vizier import Vizier
from astropy.table import Table
from scipy.stats import norm, t, skew, kurtosis, pearsonr, spearmanr
from IPython.display import display

ordner = Path("Shen2008_Auswertung")
ordner.mkdir(exist_ok=True)


## 1. Daten laden
Automatisch Csigma und logM verwenden. Fehlende Werte und nichtpositive Dispersionen entfernen. Weitere Qualitätsfilter werden nicht angewandt.

In [ ]:
viz = Vizier(columns=["**"], row_limit=-1, vizier_server="vizier.cfa.harvard.edu")
viz.TIMEOUT = 120
tabellen = viz.get_catalogs("J/AJ/135/928/table1")
if not len(tabellen):
    raise RuntimeError("Keine Daten erhalten.")
tabelle = tabellen[0]
def zahlen(v):
    return np.ma.asarray(v, dtype=float).filled(np.nan)
s0, m0 = zahlen(tabelle["Csigma"]), zahlen(tabelle["logM"])
ok = np.isfinite(s0) & np.isfinite(m0) & (s0 > 0)
sigma, logM = s0[ok], m0[ok]
masse = 10.0**logM
N = len(masse)
if N < 4:
    raise ValueError("Zu wenige gültige Daten.")
print(f"Katalogeinträge: {len(tabelle)}; gültig: {N}; ausgeschlossen: {len(tabelle)-N}")


## 2. Statistische Kennzahlen
Varianzen haben quadrierte Einheiten; Schiefe und Kurtosis sind dimensionslos. Der Standardfehler beschreibt den Mittelwert, die Standardabweichung die Streuung der Einzelwerte.

In [ ]:
namen = ["Mittelwert", "Median", "Varianz (N)", "Standardabweichung (N)",
         "Varianz (N-1)", "Standardabweichung (N-1)", "Standardfehler",
         "Minimum", "Maximum", "25%-Quantil", "75%-Quantil",
         "Interquartilsabstand", "Schiefe", "Exzess-Kurtosis"]
def stats(v):
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    return [v.mean(), med, v.var(), v.std(), v.var(ddof=1), v.std(ddof=1),
            v.std(ddof=1)/np.sqrt(len(v)), v.min(), v.max(), q1, q3, q3-q1,
            skew(v,bias=False), kurtosis(v,fisher=True,bias=False)]
statistik = Table([namen, stats(sigma), stats(masse), stats(logM)],
                 names=["Kennzahl","sigma [km/s]","M [Msun]","logM [dex]"])
for k in statistik.colnames[1:]:
    statistik[k].format = ".6g"
display(statistik)
print(f"Geometrisches Massenmittel: {10**logM.mean():.5g} Msun")
statistik.write(ordner/"Statistik.csv", format="ascii.csv", overwrite=True)


## 3. Konfidenzintervalle der Mittelwerte
Student-t-Intervalle sind bei unabhängigen, identisch normalverteilten Beobachtungen exakt. Für schiefe Verteilungen sind sie nur eine Näherung, besonders auf dem extremen 5σ-Niveau. Die linearen Massen sind stark schief verteilt. Messsystematik und Auswahlverzerrung sind nicht enthalten.


In [ ]:
rows = []
for name,v,unit in [("sigma",sigma,"km/s"),("M",masse,"Msun"),("logM",logM,"dex")]:
    se = v.std(ddof=1)/np.sqrt(N)
    for niveau,alpha in [("95%",.05),("2sigma",2*norm.sf(2)),("5sigma",2*norm.sf(5))]:
        rand = t.isf(alpha/2,N-1)*se
        rows.append((name,niveau,100*(1-alpha),v.mean()-rand,v.mean()+rand,unit))
intervalle = Table(rows=rows,names=["Größe","Niveau","Konfidenz [%]","Unten","Oben","Einheit"])
for k in ["Unten","Oben"]:
    intervalle[k].format = ".6g"
display(intervalle)
intervalle.write(ordner/"Konfidenzintervalle.csv",format="ascii.csv",overwrite=True)


## 4. Lineare Anpassung und Residuen
x = log₁₀(σ/200), y = log₁₀(M/M☉), Modell y = a + bx.
Ungewichtete kleinste Quadrate entsprechen MLE bei unabhängigen gaußförmigen Abweichungen gleicher Varianz und als fehlerfrei behandelten x-Werten. Die Parameterfehler gelten für dieses vereinfachte Modell.


In [ ]:
x,y = np.log10(sigma/200),logM
X = np.column_stack([np.ones(N),x])
p,_,rang,_ = np.linalg.lstsq(X,y,rcond=None)
if rang != 2:
    raise ValueError("Anpassung nicht eindeutig.")
a,b = p
modell = X@p
residuen = y-modell
sse = np.sum(residuen**2)
mu,s_mle = norm.fit(residuen)
se = np.sqrt(np.diag((sse/(N-2))*np.linalg.inv(X.T@X)))
rand = t.ppf(.975,N-2)*se
fit = Table([["a","b"],p,se,p-rand,p+rand],
            names=["Parameter","Wert","Standardfehler","95% unten","95% oben"])
display(fit)
print(f"logM = {a:.4f} + {b:.4f} log10(sigma/200)")
print(f"Residuenmittel: {mu:.8f} dex")
print(f"Residualvarianz (N): {sse/N:.5f} dex²")
print(f"Residualstreuung (N): {s_mle:.5f} dex")
print(f"Residualstreuung (N-2): {np.sqrt(sse/(N-2)):.5f} dex")
print(f"Pearson r (log): {pearsonr(x,y)[0]:.4f}")
print(f"Spearman r: {spearmanr(x,y)[0]:.4f}")
print(f"R²: {1-sse/np.sum((y-y.mean())**2):.4f}")
print(f"Residuen-Schiefe: {skew(residuen,bias=False):.4f}")
print(f"Residuen-Exzess-Kurtosis: {kurtosis(residuen,fisher=True,bias=False):.4f}")
fit.write(ordner/"Anpassung.csv",format="ascii.csv",overwrite=True)


## 5. Grafiken und Datenexport

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(17,5))
for achse,v,label in [(ax[0],sigma,r"$\sigma_\star$ [km/s]"),
                       (ax[1],logM,r"$\log_{10}(M/M_\odot)$")]:
    achse.hist(v,bins="fd",color="teal",edgecolor="white",alpha=.7)
    achse.axvline(v.mean(),color="black",label="Mittelwert")
    achse.axvline(np.median(v),color="darkorange",ls="--",label="Median")
    achse.set(xlabel=label,ylabel="Anzahl")
    achse.legend()
ax[2].scatter(sigma,masse,s=16,alpha=.45,color="teal")
ax[2].set(xscale="log",yscale="log",xlabel=r"$\sigma_\star$ [km/s]",
          ylabel=r"$M$ [$M_\odot$]",title=f"Masse–Sigma, N={N}")
for achse in ax:
    achse.grid(alpha=.2)
fig.tight_layout()
fig.savefig(ordner/"Verteilungen_und_Relation.png",dpi=300)
plt.show()

fig,ax = plt.subplots(1,3,figsize=(17,5))
xx = np.linspace(x.min(),x.max(),300)
ax[0].scatter(x,y,s=16,alpha=.45,color="teal")
ax[0].plot(xx,a+b*xx,color="black",label=f"b={b:.3f}")
ax[0].set(xlabel=r"$\log_{10}(\sigma/200)$",ylabel=r"$\log_{10}(M/M_\odot)$",
          title="Lineare Anpassung")
ax[0].legend()
ax[1].scatter(x,residuen,s=16,alpha=.45,color="teal")
ax[1].axhline(0,color="black",ls="--")
ax[1].set(xlabel=r"$\log_{10}(\sigma/200)$",ylabel="Residuum [dex]",title="Residuen")
ax[2].hist(residuen,bins="fd",density=True,color="teal",alpha=.6,edgecolor="white")
if s_mle > 0:
    rr = np.linspace(residuen.min(),residuen.max(),400)
    ax[2].plot(rr,norm.pdf(rr,mu,s_mle),color="black",label=f"Breite={s_mle:.3f} dex")
    ax[2].legend()
ax[2].set(xlabel="Residuum [dex]",ylabel="Dichte [1/dex]",title="Residualverteilung")
for achse in ax:
    achse.grid(alpha=.2)
fig.tight_layout()
fig.savefig(ordner/"Relation_und_Residuen.png",dpi=300)
plt.show()

ids = np.asarray(tabelle["SDSS"])[ok].astype(str)
daten = Table([ids,sigma,logM,masse,modell,residuen],
              names=["SDSS","sigma_km_s","logM_dex","M_Msun","Modell_logM","Residuum_dex"])
daten.write(ordner/"Daten_mit_Residuen.csv",format="ascii.csv",overwrite=True)
print(f"Alle Ausgaben: {ordner.resolve()}")


## Interpretation und Grenzen
- Positives Residuum: größere Masse als das Modell vorhersagt.
- Der Residuenmittelwert ist bei einer Anpassung mit freiem Achsenabschnitt automatisch nahezu null. Er liefert keinen unabhängigen Test der Literaturrelation.
- Für einen mittleren Offset zur Literatur braucht man eine extern festgelegte Referenzrelation.
- Die Residualstreuung enthält Messfehler und ist keine isolierte intrinsische Streuung.
- Eine angepasste Gaußkurve beweist keine Normalverteilung.
- Die σ- und M-Messfehler sowie die Stichprobenauswahl sind im einfachen Fit nicht modelliert.
